# 2. Preparação dos dados e engenharia de atributos

**Projeto:** Agente Identificador de Fraudes Bancárias  
**Disciplina:** Inteligência Artificial – 7º semestre – Ciência da Computação  
**Instituição:** Universidade Presbiteriana Mackenzie – Faculdade de Computação e Informática  
**Professor:** Prof. Dr. Ivan Carlos Alcântara de Oliveira

| Integrante | RA | E-mail |
|---|---|---|
| Cheuk Ki Yu | 10419664 | cheuk.yu@mackenzista.com.br |
| Gabriela Nellessen de Sousa | 10441930 | gabrielanellessen@gmail.com |
| Milton Almeida Leoncio | 10416764 | milton.almeida@live.com |

**Síntese:** Executa src/preparar_bases.py (se necessário), lista os atributos de cada base e verifica que os atributos comportamentais usam apenas transações anteriores.

**Histórico de alterações**

| Data | Autor | Descrição |
|---|---|---|
| 2026-10-28 | Milton Almeida Leoncio | Versão inicial da etapa nos resultados preliminares (N1) |
| 2026-11-20 | Milton Almeida Leoncio | Versão final do projeto (N2) |

In [1]:
# ========================================================================================
# Projeto: Agente Identificador de Fraudes Bancárias
# Disciplina: Inteligência Artificial - 7º semestre - Ciência da Computação
# Universidade Presbiteriana Mackenzie - Faculdade de Computação e Informática (FCI)
# Professor: Prof. Dr. Ivan Carlos Alcântara de Oliveira
# Integrantes:
#   Cheuk Ki Yu                  - RA 10419664 - cheuk.yu@mackenzista.com.br
#   Gabriela Nellessen de Sousa  - RA 10441930 - gabrielanellessen@gmail.com
#   Milton Almeida Leoncio       - RA 10416764 - milton.almeida@live.com
# Arquivo: notebooks/02_preparacao_atributos.ipynb
# Síntese: Executa src/preparar_bases.py (se necessário), lista os atributos de cada base e verifica que os atributos comportamentais usam apenas transações anteriores.
# Histórico de alterações:
#   2026-10-28 - Milton Almeida Leoncio - Versão inicial da etapa nos resultados preliminares (N1)
#   2026-11-20 - Milton Almeida Leoncio - Versão final do projeto (N2)
# ========================================================================================

import os, sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))
os.environ.setdefault('FRAUDES_ATRIBUTOS', str(RAIZ / 'atributos'))  # saída de preparar_bases.py
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)
FIG = RAIZ / 'resultados'

## 2.1 Geração dos atributos
Se os parquets ainda não existirem, a célula abaixo os gera a partir das bases brutas (veja `dados/README.md`). A preparação remove número do cartão, CVV, nome e endereço.

In [2]:
import comum as c
from pathlib import Path
pasta = c.pasta_atributos()
if not (pasta / 'base_a_ibm.parquet').exists():
    import preparar_bases as pb
    pb.preparar_sparkov(RAIZ / 'dados' / 'credit_card_frauds.csv').to_parquet(pasta / 'base_b_sparkov.parquet', index=False)
    pb.preparar_ibm(RAIZ / 'dados' / 'ibm').to_parquet(pasta / 'base_a_ibm.parquet', index=False)
sorted(p.name for p in pasta.glob('*.parquet'))

['base_a_ibm.parquet',
 'base_b_sparkov.parquet',
 'pred_exp1_A.parquet',
 'pred_exp1_B.parquet',
 'pred_final_A.parquet',
 'pred_final_B.parquet']

## 2.2 Atributos de cada base
Os atributos são temporais, geográficos, de perfil e comportamentais. Os comportamentais são calculados por cartão (Base A) ou cliente (Base B), apenas com o histórico anterior. Os atributos de **perfil estático** são constantes por titular; na Base A eles foram retirados do conjunto final (Experimento 1).

In [3]:
linhas = []
for chave in ('A', 'B'):
    partes, atr = c.carregar(chave)
    final = c.atributos_finais(chave, atr)
    for a in atr:
        linhas.append({'base': c.BASES[chave]['nome'], 'atributo': a, 'descrição': c.rotulo(a),
                       'perfil estático': a in c.BASES[chave]['perfil_estatico'], 'no conjunto final': a in final})
tab = pd.DataFrame(linhas)
print(tab.groupby('base')[['no conjunto final']].sum())
tab

                  no conjunto final
base                               
Base A - IBM                     29
Base B - Sparkov                 14


,base,atributo,descrição,perfil estático,no conjunto final
0,Base A - IBM,valor,valor,False,True
1,Base A - IBM,hora,hora do dia,False,True
2,Base A - IBM,dia_semana,dia da semana,False,True
3,Base A - IBM,estorno,estorno,False,True
4,Base A - IBM,tipo_uso,tipo de uso (tarja/chip/online),False,True
5,Base A - IBM,mcc,MCC (tipo de estabelecimento),False,True
6,Base A - IBM,online,compra online,False,True
7,Base A - IBM,exterior,compra no exterior,False,True
8,Base A - IBM,erro_saldo,erro: saldo insuficiente,False,True
9,Base A - IBM,erro_pin,erro: PIN incorreto,False,True


## 2.3 Verificação de ausência de vazamento temporal
A primeira transação de cada cartão/cliente não tem histórico: o tempo desde a anterior é −1 e todo comerciante é "novo". Se algum atributo usasse transações futuras, essas condições não valeriam.

In [4]:
for chave, grupo in (('A', 'cartao'), ('B', 'cliente')):
    d = pd.read_parquet(pasta / c.BASES[chave]['arquivo'], columns=[grupo, 'ts', 'seg_desde_ultima', 'novo_comerciante', 'n_trans_24h'])
    primeira = d.groupby(grupo, sort=False).head(1)  # o parquet já está na ordem (titular, data) usada na preparação
    print(c.BASES[chave]['nome'], '| primeiras transações:', len(primeira),
          '| tempo desde a anterior = -1:', bool((primeira['seg_desde_ultima'] == -1).all()),
          '| comerciante novo:', bool((primeira['novo_comerciante'] == 1).all()),
          '| transações nas 24h anteriores = 0:', bool((primeira['n_trans_24h'] == 0).all()))

Base A - IBM | primeiras transações: 5028 | tempo desde a anterior = -1: True | comerciante novo: True | transações nas 24h anteriores = 0: True
Base B - Sparkov | primeiras transações: 187 | tempo desde a anterior = -1: True | comerciante novo: True | transações nas 24h anteriores = 0: True


In [5]:
partes, atr = c.carregar('B')
partes['treino'][['ts', 'valor', 'hora', 'categoria', 'valor_sobre_media_hist', 'n_trans_24h', 'valor_24h', 'novo_comerciante', 'is_fraud']].head()

,ts,valor,hora,categoria,valor_sobre_media_hist,n_trans_24h,valor_24h,novo_comerciante,is_fraud
0,2019-01-01 00:00:44,107.230,0,4,-1.000,0,0.000,1,0
1,2019-01-01 04:57:58,61.390,4,2,0.573,1,107.230,1,0
2,2019-01-01 06:24:47,94.480,6,2,1.121,2,168.620,1,0
3,2019-01-01 15:27:33,92.480,15,7,1.055,3,263.100,1,0
4,2019-01-01 16:26:25,1.930,16,9,0.022,4,355.580,1,0
